# 通用 HOOD 推理：动作、SMPL 体型、衣服与 checkpoint

按顺序运行各单元。通常只需修改第 1、2 个代码单元。

- 输入为 **标准 SMPL、轴角弧度、米制、SMPL 坐标系**。支持 `.pkl` 字典、数值 `.npz`，以及完整 pose 的数值 `.npy`。
- 分字段格式：`body_pose[T,69]`、`global_orient[T,3]`、`transl[T,3]`、`betas[10]`；完整 pose 格式：`poses/pose[T,72]` 或 `[T,24,3]`。
- 可以用一份动作搭配另一份 betas。每次 rollout 使用固定的 10 维体型，动作至少 3 帧。
- SMPL-H/SMPL-X、旋转矩阵、四元数、BVH 必须先明确转换，不能直接截断冒充标准 SMPL。
- checkpoint 使用项目兼容的 HOOD 网络；优先读取保存的完整网络配置。模型权重形状不匹配时会报错，不会静默跳过。
- 衣服选择 `garments_dict.pkl` 中已有、带 LBS 和 coarse graph 信息的模板，例如 `tshirt`、`longskirt`。任意 OBJ 需要先完成模板构建。
- 本 notebook 只推理，不执行 backward/Adam；输出使用新的唯一目录，不覆盖动作输入或训练记录。

**内核**：选择服务器 `hood_clean` Python。首次使用可在终端执行：
```bash
/home/zhudf/miniconda3/envs/hood_clean/bin/python -m ipykernel install --user --name hood_clean --display-name 'Python (hood_clean)'
```
修改项目路径、数据路径或 GPU 后请重启内核，从头运行。只读取自己信任的 PKL 和 checkpoint。

In [ ]:
# 1. 环境：必须在导入 torch 和 HOOD 模块之前设置。
import os
import sys
from pathlib import Path

PROJECT_DIR = Path('/home/zhudf/projects/HOOD')
DATA_DIR = Path('/home/zhudf/data/hood_data/hood_data')
PHYSICAL_GPU = '0'          # 物理 GPU 编号；设为空字符串时使用 CPU

if not (PROJECT_DIR / 'train_baseline.py').is_file():
    raise FileNotFoundError('请把 PROJECT_DIR 设置为 HOOD 项目根目录')
if 'utils.defaults' in sys.modules:
    raise RuntimeError('HOOD 路径已被缓存；请重启内核后从本单元开始')
os.environ['HOOD_PROJECT'] = str(PROJECT_DIR)
os.environ['HOOD_DATA'] = str(DATA_DIR)
os.environ['CUDA_VISIBLE_DEVICES'] = PHYSICAL_GPU
os.environ['OMP_NUM_THREADS'] = os.environ['MKL_NUM_THREADS'] = '1'
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)
print('Python:', sys.executable)
print('Project:', PROJECT_DIR)

In [ ]:
# 2. 推理参数：换模型、动作、体型和衣服主要修改这里。
CHECKPOINT = DATA_DIR / 'experiments/modified_contact_relaxed_barrier_seed57_20261011T084658Z_fc628647/checkpoints/step_0000060000.pth'
MOTION_FILE = DATA_DIR / 'vto_dataset/smpl_parameters/tshirt_shape00_01_01.pkl'
GARMENT = 'longskirt'       # 'tshirt' / 'longskirt' / 其他已建好的模板

# None：沿用动作中的 betas。也可填 [0.0] * 10 或 betas 文件路径。
# 体型文件支持 .npy[10]，或包含 betas 的 .npz/.pkl 字典。
BETAS_OVERRIDE = None
SMPL_MODEL = None          # None 沿用 checkpoint；也可填绝对 .pkl 路径
GARMENT_DICT = None        # None 沿用 checkpoint；也可指定自己的兼容模板字典

# 旧官方 checkpoint 没有 config 时使用此 YAML；有配置时优先用 checkpoint 配置。
FALLBACK_CONFIG = 'postcvpr'
EXPERIMENT_LABEL = None    # None 自动取训练实验名；无训练名时取 checkpoint 文件名
SEED = 57                 # 推理图中心等随机过程的种子；文件名 seed 优先取训练 seed
DEVICE = 'cuda:0' if PHYSICAL_GPU else 'cpu'

SOURCE_FPS = 30.0          # 输入动作实际采样帧率；不自动把动作重采样为 30 fps
FRAME_START = 0            # 起始原始帧，含此帧
FRAME_STOP = None          # 终止原始帧，不含此帧；None 到结尾
FRAME_STRIDE = 1           # 每几帧取一帧；有效帧率 = SOURCE_FPS / FRAME_STRIDE
MAX_STEPS = -1             # -1 完整 rollout；调试先设 3，输入将只构建所需帧
SEPARATE_ARMS = False      # 原 HOOD 的手臂外展修正，会改变输入动作，默认关闭

MATERIAL = dict(density=0.20022, lame_mu=23600.0, lame_lambda=44400.0,
                bending_coeff=3.9625778333333325e-05)
OUTPUT_ROOT = DATA_DIR / 'inference'
SAVE_VIDEO = True
VIDEO_STRIDE = 1           # 1 每帧渲染；更大值只影响视频采样，不影响保存的 NPZ
VIDEO_MAX_FRAMES = 100000

### 输入检查与输出约定

缺少平移/根朝向/体型时会明确提示并使用零值。不会自动转换单位、坐标轴或旋转表示。
HOOD 原始数据加载器还会把 `body_pose` 最后 6 维手部关节旋转清零；`SEPARATE_ARMS=True` 时额外外展双臂。
换男性/中性 SMPL 文件时，需要模板 LBS 与其骨架/拓扑兼容；改变体型不会自动把衣服静止尺寸缩放成完全合身。

保存文件名：`实验名_seed训练种子_动作文件名_衣服_rollout.npz`。同时保存规范化输入、配置、输入/checkpoint 哈希和源代码版本。

In [ ]:
# 3. 标准化动作与体型；不修改输入文件。
import json
import pickle
from datetime import datetime, timezone
from uuid import uuid4
import numpy as np
import torch
from omegaconf import OmegaConf

from utils.inference_inputs import read_parameters, normalize_smpl_sequence
from utils.rollout_io import rollout_filename, safe_name
from utils.experiment import file_record, git_output, seed_all, write_json

CHECKPOINT, MOTION_FILE = Path(CHECKPOINT).expanduser(), Path(MOTION_FILE).expanduser()
if not CHECKPOINT.is_file():
    raise FileNotFoundError(CHECKPOINT)
if not np.isfinite(SOURCE_FPS) or SOURCE_FPS <= 0:
    raise ValueError('SOURCE_FPS 必须为正数')
if not isinstance(MAX_STEPS, int) or MAX_STEPS == 0 or MAX_STEPS < -1:
    raise ValueError('MAX_STEPS 必须为 -1 或正整数')
if DEVICE.startswith('cuda') and not torch.cuda.is_available():
    raise RuntimeError('当前内核无法访问 CUDA；请选择 hood_clean 内核/GPU，或改用 CPU')
torch.set_num_threads(1)
seed_all(SEED)
prepared, source_indices, assumptions = normalize_smpl_sequence(
    read_parameters(MOTION_FILE), BETAS_OVERRIDE, FRAME_START, FRAME_STOP, FRAME_STRIDE)
if MAX_STEPS > 0:
    source_indices = source_indices[:MAX_STEPS + 2]
    for key in ('body_pose', 'global_orient', 'transl'):
        prepared[key] = prepared[key][:MAX_STEPS + 2]
EFFECTIVE_FPS = SOURCE_FPS / FRAME_STRIDE
print('规范化输入:', {k: v.shape for k, v in prepared.items()})
print('最多预测帧数:', len(source_indices) - 2, '有效 fps:', EFFECTIVE_FPS)
for note in assumptions:
    print('输入处理:', note)

In [ ]:
# 4. 读取 checkpoint 配置，并构造兼容的模型、衣服和 SMPL 数据加载器。
from utils.arguments import load_params, create_runner
from utils.validation import Config as ValidationConfig
from utils.validation import update_config_for_validation, create_one_sequence_dataloader
from utils.common import move2device

saved = torch.load(CHECKPOINT, map_location='cpu')
stored = OmegaConf.create(saved.get('config', {}))
config_name = stored.get('config') or FALLBACK_CONFIG
modules, cfg = load_params(config_name, overrides=OmegaConf.merge(stored, {'device': DEVICE}))
if list(cfg.runner) != ['postcvpr'] or list(cfg.dataloader.dataset) != ['postcvpr']:
    raise ValueError('本 notebook 的标准 SMPL 流程使用 postcvpr runner/dataset；其他接口需适配')
body_model = SMPL_MODEL or cfg.dataloader.dataset.postcvpr.smpl_model
garment_dict = GARMENT_DICT or cfg.dataloader.dataset.postcvpr.garment_dict_file
for asset in (body_model, garment_dict):
    path = DATA_DIR / 'aux_data' / asset
    if not path.is_file():
        raise FileNotFoundError(path)
with (DATA_DIR / 'aux_data' / garment_dict).open('rb') as stream:
    available_garments = pickle.load(stream)
if GARMENT not in available_garments:
    raise ValueError(f'找不到衣服 {GARMENT}，可选：{list(available_garments)}')

vcfg = ValidationConfig(garment_dict_file=str(garment_dict), smpl_model=str(body_model),
                        separate_arms=SEPARATE_ARMS, random_betas=False, zero_betas=False,
                        restpos_scale=1.0, **MATERIAL)
cfg = update_config_for_validation(cfg, vcfg)
cfg.dataloader.dataset.postcvpr.betas_file = None  # 使用当前输入/覆盖的 betas
cfg.runner.postcvpr.regular_ts = 1.0 / EFFECTIVE_FPS
_, runner, _ = create_runner(modules, cfg, create_aux_modules=False)
runner.load_state_dict(saved.get('training_module', saved), strict=True)
runner.eval()
runner.cloth_obj.cache.clear()

training_seed = int(saved.get('seed', stored.get('seed', SEED)))
label = EXPERIMENT_LABEL or stored.get('experiment', {}).get('name') or CHECKPOINT.stem
identity = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:8]
RUN_DIR = Path(OUTPUT_ROOT) / f'{safe_name(label)}_seed{training_seed}_{identity}'
RUN_DIR.mkdir(parents=True, exist_ok=False)
INPUT_FILE = RUN_DIR / 'normalized_smpl.pkl'
with INPUT_FILE.open('xb') as stream:
    pickle.dump(prepared, stream)
loader = create_one_sequence_dataloader(INPUT_FILE, GARMENT, modules, cfg)
sequence = move2device(next(iter(loader)), DEVICE)
OmegaConf.save(cfg, RUN_DIR / 'inference_config.yaml', resolve=True)
NPZ_PATH = RUN_DIR / rollout_filename(label, training_seed, MOTION_FILE.stem, GARMENT)
record = dict(checkpoint=file_record(CHECKPOINT), motion=file_record(MOTION_FILE),
              normalized_input=file_record(INPUT_FILE), training_commit=saved.get('git_commit'),
              inference_commit=git_output(PROJECT_DIR, 'rev-parse', 'HEAD'),
              git_status=git_output(PROJECT_DIR, 'status', '--porcelain'),
              training_seed=training_seed, inference_seed=SEED, garment=GARMENT,
              smpl_model=file_record(DATA_DIR / 'aux_data' / body_model),
              garment_dict=file_record(DATA_DIR / 'aux_data' / garment_dict),
              source_fps=SOURCE_FPS, effective_fps=EFFECTIVE_FPS,
              selected_source_frames=source_indices.tolist(), betas=prepared['betas'].tolist(),
              preprocessing=dict(zero_last_six_body_pose=True, separate_arms=SEPARATE_ARMS),
              assumptions=assumptions, python=sys.version, torch=torch.__version__,
              cuda=torch.version.cuda, device=DEVICE, status='prepared')
write_json(RUN_DIR / 'metadata.json', record)
print('RUN_DIR =', RUN_DIR)
print('衣服节点数:', sequence['cloth'].pos.shape[0], '人体节点数:', sequence['obstacle'].pos.shape[0])

### Rollout 与时间推进

前两个输入帧作为上下文，因此 N 个输入帧产生 N−2 个预测状态。第一个预测状态按 HOOD 的初始化步处理（默认 1/3 秒），第二步按原流程重置初始速度，后续采用 `1 / EFFECTIVE_FPS`。
每帧衣服预测会作为下一帧的状态，不会逐帧回到初始衣服，也不会执行优化器更新。
`bare=True` 跳过物理评价项，是推理轨迹，不是统一验证指标；需要正式比较请用 `validate_experiment.py`。
修改接触损失通过训练后的网络权重影响推理；推理并不会额外运行一次 barrier 优化/CCD。

In [ ]:
# 5. 执行推理，保存完整轨迹及源动作帧对应关系。
try:
    with torch.no_grad():
        result = runner.valid_rollout(sequence, n_steps=MAX_STEPS, bare=True)
    if not np.isfinite(result['pred']).all():
        raise FloatingPointError('预测出现非有限值，请检查输入姿态、单位、材料与模型适用范围')
    frames = len(result['pred'])
    timesteps = np.full(frames, 1.0 / EFFECTIVE_FPS, dtype=np.float32)
    timesteps[0] = cfg.runner.postcvpr.initial_ts
    with NPZ_PATH.open('xb') as stream:
        np.savez_compressed(stream, pred=result['pred'], obstacle=result['obstacle'],
                            cloth_faces=result['cloth_faces'], obstacle_faces=result['obstacle_faces'],
                            source_frame_indices=source_indices[2:2 + frames],
                            timestep_s=timesteps, fps=np.array(EFFECTIVE_FPS))
    record.update(status='completed', predicted_frames=frames, rollout=str(NPZ_PATH))
except BaseException as error:
    record.update(status='failed', error=f'{type(error).__name__}: {error}')
    raise
finally:
    write_json(RUN_DIR / 'metadata.json', record)
print('预测形状 [帧数, 衣服顶点数, 3]:', result['pred'].shape)
print('NPZ:', NPZ_PATH)

In [ ]:
# 6. 可选视频：无须 OpenGL，也不修改 NPZ。长序列渲染可能比推理慢。
# VIDEO_STRIDE 只减少渲染帧数，并相应降低播放 fps，保持常规动作速度。
# 初始化帧按普通视频帧播放，视频不是其 1/3 秒松弛时间的精确重建。
from render_baseline import render_rollout
VIDEO_PATH = NPZ_PATH.with_suffix('.mp4')
if SAVE_VIDEO:
    render_rollout(NPZ_PATH, VIDEO_PATH, stride=VIDEO_STRIDE,
                   max_frames=VIDEO_MAX_FRAMES, fps=EFFECTIVE_FPS)
    record['video'] = str(VIDEO_PATH)
    write_json(RUN_DIR / 'metadata.json', record)
    print('视频:', VIDEO_PATH)
else:
    print('已跳过视频，之后可用 render_baseline.py 渲染保存的 NPZ')

In [ ]:
# 7. 可选 notebook 内播放。服务器回退的 MPEG-4 在部分浏览器中不兼容，
# 此时下载到电脑后用 VLC 播放，或安装 imageio-ffmpeg 获取独立 H.264 编码器。
from IPython.display import Video, display
if SAVE_VIDEO and VIDEO_PATH.is_file():
    display(Video(str(VIDEO_PATH), embed=True, width=640))

### 常用修改示例

- 相同动作改成长裙：`GARMENT = 'longskirt'`。
- 相同动作换体型：`BETAS_OVERRIDE = [0.0] * 10`，或 `DATA_DIR / 'my_body_betas.npy'`。
- 换动作：`MOTION_FILE` 指向标准 SMPL `.pkl/.npz/.npy`。完整 poses 包含根旋转在前 3 维。
- 换 Baseline / Modified / 新训练模型：只改 `CHECKPOINT`；若是没有配置的旧模型，再改 `FALLBACK_CONFIG`。
- 快速调试：`MAX_STEPS = 3`，确认输出后改回 `-1`。
- 动作 60 fps、每两帧取一帧：`SOURCE_FPS = 60.0; FRAME_STRIDE = 2`，有效 30 fps。
- 训练主要使用 30 fps 和给定材料范围；其他时间步、极端体型/动作可运行不代表质量有保证。

本 notebook 不写训练 TensorBoard，因为没有训练 loss；正式验证的物理指标/TensorBoard 仍由记录实验入口生成。
每次从模型加载单元开始会建立新目录；只重复保存/渲染单元会拒绝覆盖已有文件。修改参数后应重新运行后续全部单元。